# 🚦 Real-Time Traffic Intelligence — YOLOv8n Backend
> Run each cell in order. Make sure **Runtime → Change runtime type → T4 GPU** is selected.

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────
!pip install -q ultralytics flask flask-socketio flask-cors pyngrok yt-dlp
!pip install -q eventlet
print('✅ Dependencies installed')

In [ ]:
# ── Cell 2: Verify GPU ────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

In [ ]:
# ── Cell 3: Download sample traffic video (or mount your Drive) ───────
import os

# Option A – Download a short public traffic clip via yt-dlp
# Replace the URL with any YouTube traffic cam link
VIDEO_URL = 'https://www.youtube.com/watch?v=MNn9qKG2UFI'   # NYC traffic sample
if not os.path.exists('traffic.mp4'):
    !yt-dlp -f 'bestvideo[ext=mp4][height<=720]+bestaudio[ext=m4a]/best[ext=mp4]' \
        --merge-output-format mp4 -o traffic.mp4 "$VIDEO_URL" || \
    !yt-dlp -f 'best[ext=mp4]' -o traffic.mp4 "$VIDEO_URL"

# Option B – Use a file already in your Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# VIDEO_SOURCE = '/content/drive/MyDrive/traffic.mp4'

VIDEO_SOURCE = 'traffic.mp4'
print('Video source:', VIDEO_SOURCE)

In [ ]:
# ── Cell 4: Configuration ────────────────────────────────────────────
NGROK_TOKEN   = 'YOUR_NGROK_AUTH_TOKEN'   # ← Paste from https://dashboard.ngrok.com
CONFIDENCE    = 0.35
STREAM_FPS    = 15
FRAME_WIDTH   = 960
FRAME_HEIGHT  = 540

TARGET_CLASSES = {2:'Car', 3:'Motorcycle', 5:'Bus', 7:'Truck', 0:'Person'}
CLASS_COLORS   = {
    'Car':(59,130,246), 'Motorcycle':(168,85,247),
    'Bus':(234,179,8),  'Truck':(239,68,68), 'Person':(34,197,94)
}
print('Config ready ✅')

In [ ]:
# ── Cell 5: Flask + SocketIO Server ─────────────────────────────────
import cv2, time, base64, threading, json, numpy as np
from datetime import datetime
from collections import defaultdict
from ultralytics import YOLO
from flask import Flask
from flask_socketio import SocketIO
from flask_cors import CORS
from pyngrok import ngrok

app     = Flask(__name__)
CORS(app, resources={r'/*': {'origins': '*'}})
socketio = SocketIO(app, cors_allowed_origins='*', async_mode='threading',
                    max_http_buffer_size=10*1024*1024)

model = YOLO('yolov8n.pt')
model.to('cuda')
print('YOLOv8n loaded on CUDA ✅')

state = {'counts':{}, 'history':[], 'peak':0, 'events':[], 'running':False}
lock  = threading.Lock()

def draw_boxes(frame, results):
    for box in results[0].boxes:
        cls_id = int(box.cls[0])
        if cls_id not in TARGET_CLASSES: continue
        conf = float(box.conf[0])
        if conf < CONFIDENCE: continue
        label = TARGET_CLASSES[cls_id]
        color = CLASS_COLORS.get(label,(255,255,255))
        x1,y1,x2,y2 = map(int, box.xyxy[0])
        cv2.rectangle(frame,(x1,y1),(x2,y2),color,2)
        txt = f'{label} {conf:.0%}'
        (tw,th),_ = cv2.getTextSize(txt,cv2.FONT_HERSHEY_SIMPLEX,0.5,1)
        cv2.rectangle(frame,(x1,y1-th-8),(x1+tw+6,y1),color,-1)
        cv2.putText(frame,txt,(x1+3,y1-4),cv2.FONT_HERSHEY_SIMPLEX,0.5,(255,255,255),1)
    return frame

def inference_loop():
    cap = cv2.VideoCapture(VIDEO_SOURCE)
    interval = 1.0/STREAM_FPS; last_emit = 0
    with lock: state['running'] = True
    print('🎬 Inference loop started')
    while True:
        ret, frame = cap.read()
        if not ret: cap.set(cv2.CAP_PROP_POS_FRAMES,0); continue
        frame = cv2.resize(frame,(FRAME_WIDTH,FRAME_HEIGHT))
        results = model(frame, verbose=False, device='cuda')
        counts = defaultdict(int)
        for box in results[0].boxes:
            cls_id = int(box.cls[0]); conf = float(box.conf[0])
            if cls_id in TARGET_CLASSES and conf >= CONFIDENCE:
                counts[TARGET_CLASSES[cls_id]] += 1
        total = sum(counts.values())
        now   = datetime.now().strftime('%H:%M:%S')
        evts  = []
        for cls,cnt in counts.items():
            if cls in ('Bus','Truck') and cnt > 0:
                evts.append({'time':now,'message':f\"{cnt} {cls}{'s' if cnt>1 else ''} detected\",'type':cls.lower()})
        with lock:
            state['counts']  = dict(counts)
            state['peak']    = max(state['peak'], total)
            state['events']  = (evts + state['events'])[:100]
            tick = {'time':now,'total':total,**dict(counts)}
            state['history'] = (state['history']+[tick])[-120:]
        annotated = draw_boxes(frame.copy(), results)
        _, buf = cv2.imencode('.jpg', annotated, [cv2.IMWRITE_JPEG_QUALITY, 70])
        b64 = base64.b64encode(buf).decode()
        if time.time()-last_emit >= interval:
            last_emit = time.time()
            with lock: payload = {'frame':b64,'counts':state['counts'],'total':total,
                                  'peak':state['peak'],'history':state['history'][-60:],
                                  'events':state['events'][:20],'ts':now}
            socketio.emit('traffic_data', payload)
    cap.release()

@socketio.on('connect')
def on_connect():
    print('✅ Frontend connected')
    with lock:
        if not state['running']:
            threading.Thread(target=inference_loop, daemon=True).start()

print('Server code ready ✅')

In [ ]:
# ── Cell 6: Launch server & print public URL ─────────────────────────
ngrok.set_auth_token(NGROK_TOKEN)
# Kill any existing tunnels
for t in ngrok.get_tunnels(): ngrok.disconnect(t.public_url)
tunnel = ngrok.connect(5000, 'http')
public_url = tunnel.public_url.replace('http://','https://')
print(f'\n{'='*55}')
print(f'  🌐 Ngrok URL: {public_url}')
print(f'  Paste this URL into the React dashboard (⚙ Settings)')
print(f'{'='*55}\n')
socketio.run(app, host='0.0.0.0', port=5000, allow_unsafe_werkzeug=True)